# Horseshoe Tunnel Mesh with Floor-Only Electrodes

This notebook creates a 3D horseshoe-shaped tunnel mesh using **PyGIMLi**. 

Key characteristics of this configuration:
1. **Horseshoe geometry**: The tunnel roof is a 250-degree circular arch, and the bottom is flat.
2. **Floor-only electrodes**: Electrodes are placed exclusively on the flat floor of the tunnel (5 longitudinal lines with 21 electrodes each).
3. **Safe electrode shift**: The electrodes are shifted 1 mm into the rock to prevent numerical solver crashes at the boundary.
4. **Mesh refinement**: A refined zone wraps around the tunnel, and the tunnel interior is defined as a hole (unmeshed space) to save memory and avoid high contrast instability.
5. **3D ERT Forward Simulation and Inversion**: A conductive anomaly is simulated beneath the floor, and a 3D ERT inversion is carried out.

In [7]:
import pygimli as pg
import pygimli.meshtools as mt
import pygimli.physics.ert as ert
import numpy as np

print("PyGIMLi version:", pg.__version__)

PyGIMLi version: 1.5.5


## 1. Define Parameters
We specify the tunnel dimension, electrode spacing, and the sweep angle of the horseshoe arch.

In [8]:
# Tunnel Dimensions
tunnel_radius = 3.0       # Radius of the arch in meters
tunnel_length = 30.0      # Length of the modeled tunnel domain in meters (5m from first/last electrode)

# Floor Electrode Array Parameters
n_floor_lines = 5          # Number of longitudinal electrode lines on the floor
n_floor_electrodes = 21   # Number of electrodes per line
floor_x_positions = np.linspace(-1.0, 1.0, n_floor_lines) # Spanned from X=-1 to X=1
floor_z_positions = np.linspace(5, 25, n_floor_electrodes) # Centered between Z=5 and Z=25 (5m from ends)

# Geometry definitions for the Horseshoe shape
sweep_deg = 250
start_rad = np.radians(90 - (sweep_deg / 2)) # start angle
end_rad = np.radians(90 + (sweep_deg / 2))   # end angle
angles = np.linspace(start_rad, end_rad, 21)

# Main coordinates
arch_pts = [[tunnel_radius * np.cos(a), tunnel_radius * np.sin(a)] for a in angles]
floor_y = tunnel_radius * np.sin(start_rad)
ref_arch_pts = [[4.5 * np.cos(a), 4.5 * np.sin(a)] for a in angles] # Refinement boundary at r=4.5m

# --- 1mm Safe Shift ---
# Shifting the electrodes 1mm into the rock domain (downward along Y) 
# prevents the nodes from sitting exactly on the empty boundary of the hole.
shift = 0.001
safe_floor_y = floor_y - shift

print(f"Flat floor Y-coordinate: {floor_y:.3f} m")
print(f"Safe electrode Y-coordinate: {safe_floor_y:.3f} m")

Flat floor Y-coordinate: -1.721 m
Safe electrode Y-coordinate: -1.722 m


## 2. Build 2D PLC Geometry
We create a 2D cross-section comprising the background box, the refinement shell, the horseshoe tunnel, and a target anomaly below the floor.

In [9]:
# 2D Surrounding Rock Domain (Reduced to 30x30m square for speed and memory efficiency)
domain_width = 30.0
domain_2d = mt.createPolygon([[-domain_width/2, -domain_width/2], 
                             [domain_width/2, -domain_width/2], 
                             [domain_width/2, domain_width/2], 
                             [-domain_width/2, domain_width/2]], 
                             isClosed=True)

# Refinement shell polygon
ref_shell_2d = mt.createPolygon(ref_arch_pts, isClosed=True)

# Tunnel horseshoe polygon (isClosed=True connects the ends to form the flat floor)
tunnel_2d = mt.createPolygon(arch_pts, isClosed=True)

# A 3x3m anomaly box centered below the tunnel floor
anom_top = floor_y - 1.0
anom_bot = floor_y - 4.0
anomaly_2d = mt.createPolygon([[-1.5, anom_top], [1.5, anom_top], 
                               [1.5, anom_bot], [-1.5, anom_bot]], isClosed=True)

# Merge all polygons together
plc_2d = domain_2d + ref_shell_2d + tunnel_2d + anomaly_2d

# Insert the electrode nodes onto the flat floor (safe shifted positions)
for x in floor_x_positions:
    plc_2d.createNode([x, safe_floor_y])

# Define region markers (control cell sizes in 2D meshing)
# MADE COARSER: Increased the maximum area bounds for all zones to speed up the meshing and inversion
plc_2d.addRegionMarker([0, 10.0], marker=1, area=50.0)             # Marker 1: Background Rock (coarser)
plc_2d.addRegionMarker([0, 3.75], marker=2, area=2.0)              # Marker 2: Refined Rock zone (coarser)
plc_2d.addRegionMarker([0, anom_top - 1.5], marker=3, area=2.0)    # Marker 3: Anomaly zone (coarser)

# Define the tunnel interior as an unmeshed empty hole
plc_2d.addHoleMarker([0, 1.0])

print("2D PLC successfully built.")

2D PLC successfully built.


## 3. Create 2D Mesh & Extrude to 3D
We generate the triangular 2D mesh, then extrude it along the Z-axis to build a 3D prism mesh. We place strict slices at the Z coordinates of the electrodes.

In [10]:
print("Generating 2D cross-section mesh...")
mesh_2d = mt.createMesh(plc_2d, quality=34.0)

# COARSER: Reduced the baseline Z-slices from 25 to 10 to make 3D prisms longer along Z
base_z_slices = np.linspace(0, tunnel_length, 10)
anomaly_z_start, anomaly_z_end = 13.5, 16.5

all_z_targets = np.concatenate((base_z_slices, floor_z_positions, [anomaly_z_start, anomaly_z_end]))
z_slices = np.unique(np.sort(all_z_targets))
z_slices = z_slices[(z_slices >= 0) & (z_slices <= tunnel_length)]

print("Extruding 2D mesh to 3D... (this might take a few seconds)")
mesh = mt.extrudeMesh(mesh_2d, a=z_slices)

# Localize the anomaly along the Z-axis (making it 3D)
for cell in mesh.cells():
    if cell.marker() == 3:
        z_center = cell.center()[2]
        if z_center < anomaly_z_start or z_center > anomaly_z_end:
            cell.setMarker(1)

print(f"3D Mesh successfully generated!")
print(f"Mesh Nodes: {mesh.nodeCount()}")
print(f"Mesh Cells (Prisms): {mesh.cellCount()}")

# Export the mesh to VTK for Paraview visualization
mesh.exportVTK("horseshoe_tunnel_floor_only.vtk")
print("Mesh exported to 'horseshoe_tunnel_floor_only.vtk'")

Generating 2D cross-section mesh...
Extruding 2D mesh to 3D... (this might take a few seconds)
3D Mesh successfully generated!
Mesh Nodes: 34875
Mesh Cells (Prisms): 61710
Mesh exported to 'horseshoe_tunnel_floor_only.vtk'


## 4. Set Up Floor-Only ERT Scheme & Forward Simulation
We create a RAM-safe line-by-line Gradient measurement scheme using only the 105 floor electrodes.

In [11]:
print("Generating ERT scheme...")
all_lines = []

# Define the 5 floor lines of 21 electrodes each
for x in floor_x_positions:
    line = [[x, safe_floor_y, z] for z in floor_z_positions]
    all_lines.append(line)

# Build the main data container
scheme = pg.DataContainerERT()
flat_sensors = [pos for line in all_lines for pos in line]
for pos in flat_sensors:
    scheme.createSensor(pos)

# Build measurements line by line
data_idx = 0
offset = 0
for line_sensors in all_lines:
    ls = ert.createData(elecs=line_sensors, schemeName='gr')
    for i in range(ls.size()):
        scheme.createFourPointData(data_idx, 
                                   int(ls('a')[i]) + offset,
                                   int(ls('b')[i]) + offset,
                                   int(ls('m')[i]) + offset,
                                   int(ls('n')[i]) + offset)
        data_idx += 1
    offset += len(line_sensors)

print(f"Total Electrodes: {len(flat_sensors)}")
print(f"Total Measurements: {scheme.size()}")

# Forward simulation resistivities
rhomap = [
    [1, 1000.0],  # Background Rock
    [2, 1000.0],  # Refined Rock zone
    [3, 10.0],    # Conductive Anomaly
    [0, 1000.0]   # Outer space boundary (if any)
]

print("Running Forward ERT Simulation...")
data = ert.simulate(mesh, scheme=scheme, res=rhomap, noiseLevel=0.03, noiseAbs=1e-4, seed=42)

# Clean up invalid data points
data.markInvalid(data("rhoa") <= 0)
data.removeInvalid()
print(f"Simulation completed. Usable data points: {data.size()}")

# Save data
data.save("simulated_ert_floor_only.dat")
print("Simulated data saved to 'simulated_ert_floor_only.dat'")

03/07/26 - 15:41:25 - pyGIMLi - INFO - Cache c:\Users\Marios\anaconda3\envs\pg2\Lib\site-packages\pygimli\physics\ert\ert.py:createGeometricFactors restored (0.0s x 1): C:\Users\Marios\AppData\Roaming\pygimli\Cache\6038059019687349374
03/07/26 - 15:41:25 - pyGIMLi - INFO - Cache c:\Users\Marios\anaconda3\envs\pg2\Lib\site-packages\pygimli\physics\ert\ert.py:createGeometricFactors restored (0.0s x 1): C:\Users\Marios\AppData\Roaming\pygimli\Cache\8816690558797789313
03/07/26 - 15:41:25 - pyGIMLi - INFO - Cache c:\Users\Marios\anaconda3\envs\pg2\Lib\site-packages\pygimli\physics\ert\ert.py:createGeometricFactors restored (0.0s x 1): C:\Users\Marios\AppData\Roaming\pygimli\Cache\9175268923419565571
03/07/26 - 15:41:25 - pyGIMLi - INFO - Cache c:\Users\Marios\anaconda3\envs\pg2\Lib\site-packages\pygimli\physics\ert\ert.py:createGeometricFactors restored (0.0s x 1): C:\Users\Marios\AppData\Roaming\pygimli\Cache\4551234312312729669
03/07/26 - 15:41:25 - pyGIMLi - INFO - Cache c:\Users\Marios

Generating ERT scheme...
Total Electrodes: 105
Total Measurements: 420
Running Forward ERT Simulation...


03/07/26 - 15:41:25 - pyGIMLi - WARNING - parseMapToCellArray: cannot find marker 0 within mesh.
03/07/26 - 15:41:25 - pyGIMLi - INFO - Calculate geometric factors.
03/07/26 - 15:42:02 - pyGIMLi - INFO - Data error estimate (min:max)  0.03000006620182737 : 0.030021182144905748


Simulation completed. Usable data points: 420
Simulated data saved to 'simulated_ert_floor_only.dat'


## 5. 3D Inversion
We hide the anomaly from the inversion mesh (setting marker 3 back to background marker 2) and perform the ERT inversion.

In [12]:
print("Preparing inversion mesh...")
data=ert.load("simulated_ert_floor_only.dat")
inv_mesh = pg.Mesh(mesh)
for cell in inv_mesh.cells():
    if cell.marker() == 3:
        cell.setMarker(2)

print("Starting 3D Inversion... (monitoring RAM is recommended)")
mgr = ert.ERTManager()
inv_res = mgr.invert(data, mesh=inv_mesh, lam=20, verbose=True,h2=False)

# Export inversion results to VTK
inv_mesh.exportVTK("final_inverted_tunnel_floor_only.vtk")
print("Inversion successful! Results saved to 'final_inverted_tunnel_floor_only.vtk'")

Preparing inversion mesh...
Starting 3D Inversion... (monitoring RAM is recommended)


03/07/26 - 15:42:04 - pyGIMLi - INFO - Found 2 regions.
03/07/26 - 15:42:04 - pyGIMLi - INFO - Region with smallest marker set to background (marker=1)
03/07/26 - 15:42:04 - pyGIMLi - INFO - Creating forward mesh from region infos.
03/07/26 - 15:42:07 - pyGIMLi - INFO - Creating refined mesh (H2) to solve forward task.
03/07/26 - 15:42:12 - pyGIMLi - INFO - Mesh for forward task: Mesh: Nodes: 262727 Cells: 493680 Boundaries: 636908
03/07/26 - 15:42:19 - pyGIMLi - INFO - Use median(data values)=963.0720269943175
03/07/26 - 15:42:19 - pyGIMLi - INFO - Created startmodel from forward operator:42736, min/max=963.072027/963.072027
03/07/26 - 15:42:19 - pyGIMLi - INFO - Starting inversion.


fop: <pygimli.physics.ert.ertModelling.ERTModelling object at 0x0000023203DFFD80>
Data transformation: Logarithmic LU transform, lower bound 0.0, upper bound 0.0
Model transformation: Logarithmic transform
min/max (data): 277/9533
min/max (error): 3%/3%
min/max (start model): 963/963
--------------------------------------------------------------------------------


RuntimeError: ./core/src/bert/bertJacobian.cpp:289		void GIMLI::createSensitivityCol_(Matrix<T>&, const Mesh&, const DataContainerERT&, const Matrix<T>&, const RVector&, const RVector&, std::vector<std::pair<long long unsigned int, long long unsigned int> >&, uint, bool) [with ValueType = double; RVector = Vector<double>; uint = unsigned int]  potential matrix rowsize to small.104 < 105
